# Minimal Working Drowsiness Baseline (Local HF Parquet)
Loads your local datasets from `data/raw/*/data/*.parquet` and trains a tiny MobileNetV2 run.

In [ ]:
from pathlib import Path
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset, Subset
from torchvision import transforms, models
from datasets import load_dataset

# ---------------------------
# CONFIG (minimal scale)
# ---------------------------
ROOTS = [
    "data/raw/n7i5x9__driver-drowsiness-dataset",
    "data/raw/akahana__Driver-Drowsiness-Dataset",
]
IMG_SIZE = 160
BATCH_SIZE = 16
EPOCHS = 1
MAX_TRAIN_SAMPLES = 4000
MAX_VAL_SAMPLES = 800
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
random.seed(SEED)
torch.manual_seed(SEED)

# ---------------------------
# HELPERS
# ---------------------------
def normalize_label_name(name: str):
    s = str(name).lower().replace("_", " ").replace("-", " ").strip()
    if "non drowsy" in s or "not drowsy" in s:
        return 0  # alert
    if "drowsy" in s or "sleep" in s or "fatigue" in s or "yawn" in s:
        return 1
    if "alert" in s or "awake" in s or "open" in s or "normal" in s:
        return 0
    return None

def discover_data_files(root):
    root = Path(root)
    files = {}
    for p in sorted((root / "data").glob("*.parquet")):
        split = p.name.split("-")[0]
        files.setdefault(split, []).append(str(p))
    return files

class HFDrowsinessDataset(Dataset):
    def __init__(self, split_ds, transform):
        self.ds = split_ds
        self.transform = transform

        names = split_ds.features["label"].names
        self.label_map = {i: normalize_label_name(n) for i, n in enumerate(names)}

        # keep only rows whose label maps to binary class
        self.valid_idx = [i for i in range(len(split_ds)) if self.label_map.get(int(split_ds[i]["label"])) is not None]

    def __len__(self):
        return len(self.valid_idx)

    def __getitem__(self, idx):
        row = self.ds[self.valid_idx[idx]]
        img = row["image"]
        y = self.label_map[int(row["label"])]
        x = self.transform(img.convert("RGB"))
        return x, torch.tensor(y, dtype=torch.long)

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ---------------------------
# LOAD LOCAL DATASETS
# ---------------------------
train_parts = []
val_parts = []

for root in ROOTS:
    data_files = discover_data_files(root)
    if not data_files:
        print(f"Skip missing parquet root: {root}")
        continue

    ds = load_dataset("parquet", data_files=data_files)
    print(f"Loaded {root} splits: {list(ds.keys())}")

    if "train" in ds:
        train_parts.append(HFDrowsinessDataset(ds["train"], train_tf))

    # Prefer validation when present, else test
    if "validation" in ds:
        val_parts.append(HFDrowsinessDataset(ds["validation"], eval_tf))
    elif "test" in ds:
        val_parts.append(HFDrowsinessDataset(ds["test"], eval_tf))

assert train_parts, "No train split found in local parquet datasets."
assert val_parts, "No validation/test split found in local parquet datasets."

train_ds = ConcatDataset(train_parts)
val_ds = ConcatDataset(val_parts)

if len(train_ds) > MAX_TRAIN_SAMPLES:
    train_ds = Subset(train_ds, list(range(MAX_TRAIN_SAMPLES)))
if len(val_ds) > MAX_VAL_SAMPLES:
    val_ds = Subset(val_ds, list(range(MAX_VAL_SAMPLES)))

print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ---------------------------
# MINIMAL MODEL + TRAIN
# ---------------------------
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 2)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += x.size(0)

    print(f"Epoch {epoch+1}: train_loss={running_loss/total:.4f} train_acc={correct/total:.4f}")

# ---------------------------
# QUICK EVAL
# ---------------------------
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for x, y in val_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += x.size(0)

print(f"Validation accuracy: {correct/total:.4f}")
